# 🎙️ Phase 1 — Part 2: Data Collection & Audio Clipping

---

## What this notebook does
1. **Downloads** Bangla audio from YouTube (Dhaka + Chittagong dialects)
2. **Cuts** long audio files into **10-second clips** at 16kHz mono WAV
3. **Saves everything to Google Drive** so nothing is lost if Colab disconnects

## What gets saved & where
| File | Location in Drive | Persists? |
|------|-------------------|----------|
| Full downloaded audio | `NSU_499A/raw_audio/dhaka/` or `chittagong/` | ✅ Yes |
| 10-second clips | `NSU_499A/clips/dhaka/` or `chittagong/` | ✅ Yes |
| `cut_audio.py` script | `NSU_499A/scripts/` | ✅ Yes |

## If Colab disconnects?
- Already downloaded files are safe in Drive
- Re-run only from the point where you stopped
- The clip script skips already-processed files

---

## Step 0: Mount Drive & Load Config

In [ ]:
# ════════════════════════════════════════════════════════════════
# MOUNT DRIVE & LOAD CONFIG — Run this first in EVERY notebook
# ════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import json, os
config = json.load(open('/content/drive/MyDrive/NSU_499A/config.json'))
BASE = config['base_path']
print(f'✅ Project root: {BASE}')

## Step 1: Install Required Tools

- **yt-dlp**: Downloads audio from YouTube
- **librosa**: Audio processing library
- **soundfile**: Saves WAV files

> These install into the Colab session (temporary). You need to re-run this cell each time you open a new session.

In [ ]:
!pip install -q yt-dlp librosa soundfile
print('✅ yt-dlp, librosa, soundfile installed')

## Step 2: Download Audio from YouTube

### How to use:
1. Find a YouTube video with clear Bangla speech (Dhaka or Chittagong dialect)
2. Copy the video URL
3. Paste it into the `urls` list below
4. Set the correct `dialect` ('dhaka' or 'chittagong')
5. Run the cell

### Good sources for **Dhaka dialect**:
- বাংলা নাটক (Bangla Natok/Drama)
- News: BTV, Jamuna TV, Ekattor TV
- Talk shows: ইত্যাদি, Mosharraf Karim dramas

### Good sources for **Chittagong dialect**:
- চাটগাঁইয়া নাটক / চাটগাঁইয়া কমেডি
- Chittagong local vlogs
- **Listen for 30 seconds first** — make sure they speak actual Chittagong dialect, not Standard Bangla

### ⚠️ IMPORTANT:
- Download **at least 50 videos per dialect** (target: 2-3 hours total per dialect)
- You can run this cell **multiple times** with different URLs
- Already downloaded files won't be re-downloaded

In [ ]:
# ════════════════════════════════════════════════════════════════
# DOWNLOAD AUDIO — Edit the urls list and dialect, then run
# ════════════════════════════════════════════════════════════════

dialect = 'dhaka'  # ← Change to 'chittagong' for Chittagong dialect

urls = [
    # Paste YouTube URLs here, one per line:
    # 'https://www.youtube.com/watch?v=XXXXXXXXX',
    # 'https://www.youtube.com/watch?v=YYYYYYYYY',
]

# ── Download ──
output_dir = os.path.join(BASE, f'raw_audio/{dialect}')

if len(urls) == 0:
    print('⚠️ No URLs provided! Paste YouTube URLs in the list above.')
else:
    for i, url in enumerate(urls):
        print(f'\n📥 Downloading {i+1}/{len(urls)}: {url}')
        cmd = f'yt-dlp -x --audio-format wav --audio-quality 0 -o "{output_dir}/%(title)s.%(ext)s" "{url}"'
        os.system(cmd)

    # Count downloaded files
    wav_files = [f for f in os.listdir(output_dir) if f.endswith('.wav')]
    print(f'\n✅ Total WAV files in {dialect}/: {len(wav_files)}')

### (Alternative) Download an entire YouTube Playlist

In [ ]:
# ════════════════════════════════════════════════════════════════
# DOWNLOAD PLAYLIST — Uncomment and edit, then run
# ════════════════════════════════════════════════════════════════

# dialect = 'dhaka'  # or 'chittagong'
# playlist_url = 'https://www.youtube.com/playlist?list=YOUR_PLAYLIST_ID'
# max_videos = 20  # limit number of downloads

# output_dir = os.path.join(BASE, f'raw_audio/{dialect}')
# cmd = f'yt-dlp -x --audio-format wav --audio-quality 0 --playlist-end {max_videos} -o "{output_dir}/%(playlist_index)s_%(title)s.%(ext)s" "{playlist_url}"'
# os.system(cmd)
# print('✅ Playlist download complete!')

## Step 3: Check What You've Downloaded
Run this to see how many files you have per dialect.

In [ ]:
import librosa

for d in ['dhaka', 'chittagong']:
    folder = os.path.join(BASE, f'raw_audio/{d}')
    if not os.path.exists(folder):
        print(f'📁 raw_audio/{d}/  — folder empty or missing')
        continue
    wav_files = [f for f in os.listdir(folder) if f.endswith('.wav')]
    total_sec = 0
    for wf in wav_files[:5]:  # check first 5 for duration estimate
        try:
            dur = librosa.get_duration(filename=os.path.join(folder, wf))
            total_sec += dur
        except:
            pass
    avg_dur = total_sec / max(len(wav_files[:5]), 1)
    est_total = avg_dur * len(wav_files) / 3600
    print(f'📁 raw_audio/{d}/  — {len(wav_files)} files, ~{est_total:.1f} hours estimated')

## Step 4: Cut Long Audio into 10-Second Clips

### Why 10-second clips?
- ASR models work best with short audio segments (5-15 seconds)
- Longer audio causes memory issues during training
- Each clip gets one text label (transcript)

### What this does:
- Loads each WAV file
- Resamples to **16kHz mono** (required by all ASR models)
- Cuts into 10-second non-overlapping segments
- Skips silent clips (max amplitude < 0.01)
- Saves to `clips/dhaka/` or `clips/chittagong/`

### What is saved:
- The script itself → `scripts/cut_audio.py` (permanent copy)
- All clips → `clips/{dialect}/` (permanent in Drive)

In [ ]:
# ════════════════════════════════════════════════════════════════
# SAVE cut_audio.py TO DRIVE (permanent copy)
# ════════════════════════════════════════════════════════════════

cut_audio_code = '''
import librosa
import soundfile as sf
import os

# Settings
CLIP_SEC = 10       # each clip = 10 seconds
SAMPLE_RATE = 16000  # 16kHz required by all ASR models

def cut_and_save(input_wav, output_dir, dialect, file_index=0):
    """Cut a long WAV file into 10-second clips."""
    audio, sr = librosa.load(input_wav, sr=SAMPLE_RATE, mono=True)
    os.makedirs(output_dir, exist_ok=True)
    
    clips_made = 0
    step = CLIP_SEC * SAMPLE_RATE
    
    for i, start in enumerate(range(0, len(audio) - step, step)):
        clip = audio[start : start + step]
        
        # Skip silent clips
        if clip.max() < 0.01:
            continue
        
        out_name = f"{dialect}_f{file_index:03d}_clip_{i:05d}.wav"
        sf.write(os.path.join(output_dir, out_name), clip, SAMPLE_RATE)
        clips_made += 1
    
    return clips_made
'''

script_path = os.path.join(BASE, 'scripts/cut_audio.py')
with open(script_path, 'w') as f:
    f.write(cut_audio_code)
print(f'✅ Script saved to: {script_path}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# RUN CLIPPING — Processes all raw audio → 10-sec clips
# ════════════════════════════════════════════════════════════════

import librosa
import soundfile as sf

CLIP_SEC = 10
SAMPLE_RATE = 16000

def cut_and_save(input_wav, output_dir, dialect, file_index=0):
    audio, sr = librosa.load(input_wav, sr=SAMPLE_RATE, mono=True)
    os.makedirs(output_dir, exist_ok=True)
    clips_made = 0
    step = CLIP_SEC * SAMPLE_RATE
    for i, start in enumerate(range(0, len(audio) - step, step)):
        clip = audio[start : start + step]
        if clip.max() < 0.01:
            continue
        out_name = f"{dialect}_f{file_index:03d}_clip_{i:05d}.wav"
        out_path = os.path.join(output_dir, out_name)
        # Skip if already exists (resume-safe)
        if not os.path.exists(out_path):
            sf.write(out_path, clip, SAMPLE_RATE)
        clips_made += 1
    return clips_made

# Process both dialects
for dialect in ['dhaka', 'chittagong']:
    raw_dir = os.path.join(BASE, f'raw_audio/{dialect}')
    clip_dir = os.path.join(BASE, f'clips/{dialect}')
    
    if not os.path.exists(raw_dir):
        print(f'⚠️ raw_audio/{dialect}/ not found, skipping')
        continue
    
    wav_files = sorted([f for f in os.listdir(raw_dir) if f.endswith('.wav')])
    if len(wav_files) == 0:
        print(f'⚠️ No WAV files in raw_audio/{dialect}/, skipping')
        continue
    
    print(f'\n🔪 Cutting {dialect} audio ({len(wav_files)} files)...')
    total_clips = 0
    for idx, fname in enumerate(wav_files):
        filepath = os.path.join(raw_dir, fname)
        try:
            n = cut_and_save(filepath, clip_dir, dialect, file_index=idx)
            total_clips += n
            print(f'  ✂️ {fname}: {n} clips')
        except Exception as e:
            print(f'  ❌ {fname}: Error — {e}')
    
    print(f'\n✅ {dialect}: {total_clips} total clips saved to clips/{dialect}/')

## Step 5: Verify Clips
Check how many clips you have. **Target: ~360 clips per dialect (= ~1 hour)**.

In [ ]:
for dialect in ['dhaka', 'chittagong']:
    clip_dir = os.path.join(BASE, f'clips/{dialect}')
    if os.path.exists(clip_dir):
        clips = [f for f in os.listdir(clip_dir) if f.endswith('.wav')]
        hours = len(clips) * 10 / 3600  # each clip = 10 sec
        print(f'📁 clips/{dialect}/: {len(clips)} clips (~{hours:.1f} hours)')
        if len(clips) < 360:
            print(f'   ⚠️ Need more! Target is 360+ clips (1 hour). Download more audio.')
        else:
            print(f'   ✅ Good! You have enough for training.')
    else:
        print(f'📁 clips/{dialect}/: No clips yet')

---
## ✅ Part 2 Complete!

**What is now saved in Google Drive:**
- `raw_audio/dhaka/` — Full downloaded audio files
- `raw_audio/chittagong/` — Full downloaded audio files
- `clips/dhaka/` — 10-second clips ready for transcription
- `clips/chittagong/` — 10-second clips ready for transcription
- `scripts/cut_audio.py` — The clipping script (permanent copy)

**If Colab disconnects:** All files are safe in Drive. Just re-mount and continue.

**Next → Open `Phase1_Part3_Transcription.ipynb`**

---